In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from MuyGPyS._test.sampler import print_results
from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize, L_BFGS_B_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn

# Import and setup data

In [2]:
BASE_DIR = os.path.abspath("..") + "\\"
aqi_data = np.genfromtxt(BASE_DIR + "data\\biggest_day_aqi_coords_scaled.csv", delimiter=',', skip_header=1)
print(aqi_data[:10])

[[ 0.27128141  0.75872801 53.        ]
 [ 0.33042753  0.78042228 47.        ]
 [ 0.36178224  0.76125781 69.        ]
 [ 0.35172346  0.7786855  69.        ]
 [ 0.31374196  0.77694373 48.        ]
 [ 0.2842055   0.76718129 52.        ]
 [ 0.34535027  0.77844838 65.        ]
 [ 0.33611686  0.76985651 74.        ]
 [ 0.35287642  0.76437478 63.        ]
 [ 0.36015016  0.76916901 45.        ]]


In [3]:
#randomly split the data into features and responces, and training and testing paritions.
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]
aqi_x_train, aqi_x_test, aqi_y_train, aqi_y_test = train_test_split(aqi_x, aqi_y, test_size=0.1, random_state=24)
#the below was created to try and istimate a mean functin.
aqi_y_mean = aqi_y_train.mean()
aqi_y_train = aqi_y_train - aqi_y_mean
print(aqi_y_test[:5])

[28. 31. 59. 58. 77.]


# Nearest Neighbor and Batches

In [4]:
#setup nearest nieghbors
nn_count = 30
nbrs_lookup = NN_Wrapper(aqi_x_train, nn_count, nn_method="exact", algorithm="ball_tree")


In [5]:
#for the sake of making use of batches, will use a batch value of 1500
batch_count = 1500
batch_indices, batch_nn_indices = sample_batch(
    nbrs_lookup, batch_count, len(aqi_x_train)
)

# setting and optimizing hyperparamters

In [6]:
aqi_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.1, 4.0)),
        deformation=Isotropy(
            l2,
            length_scale=Parameter("log_sample", (0.01, 2.0)),
        ),
    ),
    noise=HomoscedasticNoise("sample", (1e-6, 1.0)),
    scale=AnalyticScale(),
)

In [7]:
# #crosswise distance: distances between a point and each of it's neighbors
# #pairwise distance: the distances between all of a point's neighbors and each other. for the purpose of finding which points contribute together and should share credit.

# batch_ys and batch_nn_ys are the responces to the training points, which we need to grab.
(
    batch_crosswise_dists,
    batch_pairwise_dists,
    batch_ys,
    batch_nn_ys,
) = aqi_muygps.make_train_tensors(
    batch_indices,
    batch_nn_indices,
    aqi_x_train,
    aqi_y_train,
)
#there are the distance tensors above which store the tensors from each batch point to it's neighbors
#as well as, for each batch point, between each of the neighbors and each other.

#We don't calculate the crosswise and pairwise tensors here, because that's handled by Bayes_optimize.

In [8]:
#optimize the parameters
aqi_muygps_optimized = Bayes_optimize(
    aqi_muygps,
    batch_ys,
    batch_nn_ys,
    batch_crosswise_dists,
    batch_pairwise_dists,
    loss_fn=lool_fn,
    verbose=True,
    random_state=24,
    init_points=15,
    n_iter=25,
)

# aqi_muygps_optimized = L_BFGS_B_optimize(
#     aqi_muygps,
#     batch_ys,
#     batch_nn_ys,
#     batch_crosswise_dists,
#     batch_pairwise_dists,
#     loss_fn=lool_fn,
# )

parameters to be optimized: ['length_scale', 'smoothness', 'noise']
bounds: [[1.e-02 2.e+00]
 [1.e-01 4.e+00]
 [1.e-06 1.e+00]]
initial x0: [0.07489467 0.22386422 0.17309273]
|   iter    |  target   | length... | smooth... |   noise   |
-------------------------------------------------------------
| 1         | -6263.839 | 0.0748946 | 0.2238642 | 0.1730927 |
| 2         | -9047.993 | 1.9204344 | 2.8280969 | 0.9998672 |
| 3         | -10285.76 | 0.4479339 | 1.5081197 | 0.7398412 |
| 4         | -39336.56 | 1.9929468 | 1.3337532 | 0.1365454 |
| 5         | -16507.89 | 0.7741202 | 1.3500252 | 0.3664153 |
| 6         | -13071.94 | 1.4222066 | 3.6105554 | 0.5341159 |
| 7         | -12401.10 | 0.5021145 | 2.7200455 | 0.5617295 |
| 8         | -9864.836 | 1.0896941 | 3.5844456 | 0.8427797 |
| 9         | -11071.28 | 0.6189650 | 2.5615621 | 0.6802391 |
| 10        | -9319.909 | 1.9411508 | 3.5849118 | 0.9424259 |
| 11        | -25220.63 | 1.2880287 | 2.4971257 | 0.2276840 |
| 12        | -9850

In [9]:
#also optimize the scale
aqi_muygps_optimized = aqi_muygps_optimized.optimize_scale(
    batch_pairwise_dists,
    batch_nn_ys
)

# Inference

In [10]:
test_count = aqi_x_test.shape[0]
test_indices = np.arange(test_count)
test_nn_indices, _ = nbrs_lookup.get_nns(aqi_x_test)

In [11]:
(
    test_crosswise_dists,
    test_pairwise_dists,
    test_nn_ys,
) = aqi_muygps.make_predict_tensors(
    test_indices,
    test_nn_indices,
    aqi_x_test,
    aqi_x_train,
    aqi_y_train,
)

kcross = aqi_muygps_optimized.kernel(test_crosswise_dists)
kin = aqi_muygps_optimized.kernel(test_pairwise_dists)

In [12]:
print(test_crosswise_dists[:2])

print(test_pairwise_dists[:2])

print(test_nn_ys[:2])

[[0.00556367 0.01189713 0.0135696  0.01517876 0.01644134 0.01808074
  0.01872635 0.021606   0.02454123 0.0249527  0.0257014  0.02585181
  0.02619569 0.02652471 0.02861123 0.03376886 0.03601164 0.03670298
  0.03678366 0.03792197 0.0393825  0.04146507 0.04600683 0.04801405
  0.04864475 0.0501524  0.05041901 0.05107234 0.05248899 0.05569493]
 [0.00247994 0.00447599 0.00507091 0.0063473  0.01084798 0.01404015
  0.01428185 0.01708286 0.01835768 0.01941885 0.03076428 0.03118021
  0.03359098 0.03372247 0.03389665 0.04093959 0.04134407 0.0418963
  0.04344048 0.04422729 0.04623585 0.048146   0.04828739 0.04938913
  0.05524412 0.05607462 0.05642191 0.05656796 0.0580655  0.06345533]]
[[[0.         0.01729694 0.00811634 ... 0.05573356 0.04729086 0.05370384]
  [0.01729694 0.         0.02483299 ... 0.03991822 0.06244008 0.05831273]
  [0.00811634 0.02483299 0.         ... 0.06161852 0.03925223 0.04974439]
  ...
  [0.05573356 0.03991822 0.06161852 ... 0.         0.09205646 0.06794033]
  [0.04729086 0.

In [13]:
predictions = aqi_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
predictions = predictions + aqi_y_mean
variances = aqi_muygps_optimized.posterior_variance(kin, kcross)
#confidence interval means within 1.96 of the standard of deviation for that point.
confidence_intervals = np.sqrt(variances) * 1.96
#coverage is the proportion of posterior means (guesses) that differ from the true response by no more than the confidence interval size.
coverage = np.count_nonzero(np.abs(aqi_y_test - predictions) < confidence_intervals) / test_count

In [14]:
print(coverage)
print()
print(aqi_y_test[:5])
print()
print(predictions[:5])
print()
print(confidence_intervals[:5])
print()
print(test_count)


0.6818181818181818

[28. 31. 59. 58. 77.]

[34.87361092 35.27286571 55.53983796 49.30249309 53.11605187]

[14.22972274 13.67924904 14.03770562 13.45127726 13.94101382]

110


In [15]:
print_results(
    aqi_y_test, ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.100000,1.014995,1.000000,124.698305,15.773057,51.101938,13.989606,0.681818
